In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, r2_score, mean_absolute_error, root_mean_squared_error, mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
import tensorflow as tf
import optuna
import datetime
import os
import shutil

In [7]:
df = pd.read_csv('diamonds.csv')
print(df.columns.tolist())

categorical_features = ['cut', 'color', 'clarity']
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z']

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

X = df.drop(['price'], axis=1)
y = df['price'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_transformed = full_pipeline.fit_transform(X_train)
X_test_transformed = full_pipeline.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

print(f"Признаков: {X_train_transformed.shape[1]}, Объектов: {X_train_transformed.shape[0]}")

['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z', 'price']
Признаков: 23, Объектов: 43152


In [8]:
X_train_t = torch.tensor(X_train_transformed, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test_transformed, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32).reshape(-1, 1)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)


In [9]:
class DiamondRegressionPyTorch(nn.Module):
    def __init__(self, input_size, hidden_sizes=[64, 32, 16], dropout_rate=0.2, norm_type='batch'):
        super(DiamondRegressionPyTorch, self).__init__()
        layers = []
        prev = input_size
        
        for i, h in enumerate(hidden_sizes):
            layers.append(nn.Linear(prev, h))
            
            if norm_type == 'batch':
                layers.append(nn.BatchNorm1d(h))
            elif norm_type == 'layer':
                layers.append(nn.LayerNorm(h))
            elif norm_type == 'instance':
                layers.append(nn.InstanceNorm1d(h))
            
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev = h
        
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)

input_size = X_train_transformed.shape[1]
model_pt = DiamondRegressionPyTorch(
    input_size=input_size,
    hidden_sizes=[64, 32, 16],
    dropout_rate=0.2,
    norm_type='batch'
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)

In [10]:

class EarlyStoppingPyTorch:
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
        return self.early_stop

early_stopping_pt = EarlyStoppingPyTorch(patience=7)

In [11]:
epochs_pt = 100

for epoch in range(epochs_pt):
    model_pt.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    model_pt.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model_pt(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    val_loss /= len(test_loader)
    
    
    
    if (epoch + 1) % 10 == 0:
        print(f"Эпоха {epoch+1}/{epochs_pt}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    if early_stopping_pt(val_loss):
        print(f"Early stopping at epoch {epoch+1}")
        break
   


Эпоха 10/100, Train Loss: 0.0930, Val Loss: 0.0367
Эпоха 20/100, Train Loss: 0.0818, Val Loss: 0.0252
Early stopping at epoch 27


In [12]:

y_pred_scaled = []
y_true_scaled = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model_pt(inputs)
        y_pred_scaled.extend(outputs.numpy().flatten())
        y_true_scaled.extend(labels.numpy().flatten())

y_pred_pt = scaler_y.inverse_transform(np.array(y_pred_scaled).reshape(-1, 1)).flatten()
y_true_pt = scaler_y.inverse_transform(np.array(y_true_scaled).reshape(-1, 1)).flatten()

print("\nРезультаты PyTorch модели:")
print(f"MSE: {mean_squared_error(y_true_pt, y_pred_pt):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true_pt, y_pred_pt)):.2f}")
print(f"MAE: {mean_absolute_error(y_true_pt, y_pred_pt):.2f}")
print(f"R²: {r2_score(y_true_pt, y_pred_pt):.4f}")


Результаты PyTorch модели:
MSE: 411620.47
RMSE: 641.58
MAE: 409.92
R²: 0.9741


In [16]:

early_stopping_tf = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

def create_tf_model(input_dim, dropout_rate=0.2):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(dropout_rate),
        
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.LayerNormalization(),
        tf.keras.layers.Dropout(dropout_rate),
        
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.GroupNormalization(groups=-1),
        tf.keras.layers.Dropout(dropout_rate),
        
        tf.keras.layers.Dense(1)
    ])
    return model

model_tf = create_tf_model(input_dim=input_size, dropout_rate=0.2)

model_tf.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae', 'mse']
)

In [17]:
history = model_tf.fit(
    X_train_transformed, y_train_scaled,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_transformed, y_test_scaled),
    callbacks=[early_stopping_tf],
    verbose=1
)

Epoch 1/100
675/675 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 1.0004 - mae: 0.7610 - mse: 1.0004 - val_loss: 0.9990 - val_mae: 0.7608 - val_mse: 0.9990
Epoch 2/100
675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0002 - mae: 0.7615 - mse: 1.0002 - val_loss: 0.9989 - val_mae: 0.7501 - val_mse: 0.9989
Epoch 3/100
675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0000 - mae: 0.7606 - mse: 1.0000 - val_loss: 0.9997 - val_mae: 0.7643 - val_mse: 0.9997
Epoch 4/100
675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0001 - mae: 0.7607 - mse: 1.0001 - val_loss: 0.9998 - val_mae: 0.7647 - val_mse: 0.9998
Epoch 5/100
675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0001 - mae: 0.7608 - mse: 1.0001 - val_loss: 1.0000 - val_mae: 0.7656 - val_mse: 1.0000
Epoch 6/100
675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0002 - mae: 0.7619 - mse: 1.0002 - val_loss: 0.9986 - val_mae: 0.7543 - val_mse: 0.9986
Epoch 7/100
675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0002 - mae: 0.7609 - mse: 1.000

In [ ]:
y_pred_tf_scaled = model_tf.predict(X_test_transformed).flatten()
y_pred_tf = scaler_y.inverse_transform(y_pred_tf_scaled.reshape(-1, 1)).flatten()

print("\nРезультаты TensorFlow модели:")
print(f"MSE: {mean_squared_error(y_test, y_pred_tf):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tf)):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_tf):.2f}")
print(f"R²: {r2_score(y_test, y_pred_tf):.4f}")

338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  

Результаты TensorFlow модели:
MSE: 410230.47
RMSE: 640.49
MAE: 383.35
R²: 0.9742
